In [2]:
import pandas as pd, numpy as np, re
from scipy import stats
from sklearn.decomposition import PCA

In [13]:
# ĐỌC FILE KHẢO SÁT
FILE_XLSX = "data_2.xlsx"
df_raw = pd.read_excel(FILE_XLSX, sheet_name=0)

# Chuẩn hóa tên cột
df = df_raw.copy()
df.columns = [str(c).strip() for c in df.columns]

# Lọc các cột thang đo (đúng cấu trúc RE, EMP, DOC, SAT)
prefix = ("RE", "EMP", "DOC", "SAT")
item_cols = [c for c in df.columns if any(str(c).startswith(p) for p in prefix)]

# Ép kiểu số (Likert)
df[item_cols] = df[item_cols].apply(pd.to_numeric, errors="coerce")

# Loại dòng thiếu dữ liệu
df_items = df[item_cols].dropna().reset_index(drop=True)

print("Số mẫu hợp lệ:", len(df_items))

Số mẫu hợp lệ: 75


In [ ]:
FILE_XLSX = "data_2.xlsx"
N_FACTORS = 4          # chạy theo mô hình lý thuyết RE-EMP-DOC-SAT (đổi 3 nếu theo Kaiser)
MIN_LOADING = 0.50     # ngưỡng loading để đánh dấu biến yếu

# -------------------------
# 0) READ + AUTO RENAME (B1/B2/B3/PHẦN C -> RE/EMP/DOC/SAT)
# -------------------------
df = pd.read_excel(FILE_XLSX, sheet_name=0)
df.columns = [str(c).strip() for c in df.columns]

def pick_cols(pattern: str):
    pat = re.compile(pattern, flags=re.IGNORECASE)
    return [c for c in df.columns if pat.search(str(c))]

b1_cols  = pick_cols(r"\bB1\b")
b2_cols  = pick_cols(r"\bB2\b")
b3_cols  = pick_cols(r"\bB3\b")
sat_cols = pick_cols(r"PHẦN\s*C")

rename_map = {}
for i, c in enumerate(b1_cols,  start=1): rename_map[c] = f"RE{i}"
for i, c in enumerate(b2_cols,  start=1): rename_map[c] = f"EMP{i}"
for i, c in enumerate(b3_cols,  start=1): rename_map[c] = f"DOC{i}"
for i, c in enumerate(sat_cols, start=1): rename_map[c] = f"SAT{i}"

df2 = df.rename(columns=rename_map)
item_cols = list(rename_map.values())

df_items = df2[item_cols].apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)


In [20]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# HÀM TÍNH CRONBACH'S ALPHA
# ============================================================
def cronbach_alpha(items: pd.DataFrame) -> float:
    items = items.dropna()
    k = items.shape[1]
    if k < 2:
        return np.nan

    variances = items.var(ddof=1)
    total_score = items.sum(axis=1)
    total_variance = total_score.var(ddof=1)

    if total_variance == 0 or np.isnan(total_variance):
        return np.nan

    return (k / (k - 1)) * (1 - variances.sum() / total_variance)

# ============================================================
# Corrected Item – Total Correlation
# ============================================================
def corrected_item_total_corr(items: pd.DataFrame) -> pd.Series:
    items = items.dropna()
    total = items.sum(axis=1)
    return pd.Series({
        col: items[col].corr(total - items[col])
        for col in items.columns
    })

# ============================================================
# Alpha if Item Deleted
# ============================================================
def alpha_if_item_deleted(items: pd.DataFrame) -> pd.Series:
    return pd.Series({
        col: cronbach_alpha(items.drop(columns=[col]))
        for col in items.columns
    })

# ============================================================
# BẢNG ITEM–TOTAL STATISTICS (GIỐNG SPSS)
# ============================================================
def cronbach_item_table(items: pd.DataFrame) -> pd.DataFrame:
    items = items.dropna()

    return pd.DataFrame({
        "Corrected Item-Total Correlation":
            corrected_item_total_corr(items),
        "Cronbach's Alpha if Item Deleted":
            alpha_if_item_deleted(items)
    })

# ============================================================
# CHẠY CRONBACH CHO NHIỀU THANG ĐO + XUẤT EXCEL
# ============================================================
def run_cronbach_and_export(
    df_items: pd.DataFrame,
    scale_prefixes: dict,
    output_path="Cronbach_Alpha_Results.xlsx",
    itc_threshold=0.30
):
    """
    scale_prefixes = {
        "RE": "Kho hàng thân thiện môi trường",
        "EMP": "Nhân sự logistics xanh",
        "DOC": "Chính sách & tài liệu môi trường",
        "SAT": "Hình ảnh thương hiệu xanh"
    }
    """

    summary_rows = []
    writer = pd.ExcelWriter(output_path, engine="xlsxwriter")

    for prefix, scale_name in scale_prefixes.items():
        cols = [c for c in df_items.columns if c.startswith(prefix)]
        items = df_items[cols].dropna()

        alpha = cronbach_alpha(items)

        # ===== BẢNG ITEM–TOTAL =====
        item_table = cronbach_item_table(items)
        item_table["Kết luận"] = np.where(
            item_table["Corrected Item-Total Correlation"] < itc_threshold,
            "Loại",
            "Giữ"
        )

        item_table.index.name = "Biến quan sát"

        # Ghi sheet riêng (giống SPSS)
        item_table.to_excel(
            writer,
            sheet_name=f"{prefix}_ItemTotal"
        )

        # ===== DÒNG TỔNG HỢP =====
        summary_rows.append({
            "Thang đo": scale_name,
            "Ký hiệu": prefix,
            "Cronbach’s Alpha": round(alpha, 3),
            "Số biến quan sát": items.shape[1],
            "Số quan sát": items.shape[0],
            "Đạt yêu cầu (α ≥ 0.7)": "Đạt" if alpha >= 0.7 else "Chưa đạt"
        })

    # ===== SHEET TỔNG HỢP (GIỐNG SPSS – RELIABILITY STATISTICS) =====
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_excel(writer, sheet_name="Tong_hop_Cronbach", index=False)

    writer.close()

    print(f"✅ Đã xuất file: {Path(output_path).resolve()}")



In [38]:
# ============================================================
# EFA (1 KHUNG CODE) -> TRẢ KẾT QUẢ NGAY:
# - KMO (overall + per-item)
# - Bartlett (Chi-square, df, p-value)
# - Eigenvalues (Kaiser > 1)
# - Factor Loadings (PCA-based + Varimax)  <-- giống thực hành SPSS: PCA + Varimax
# - Communalities
#
# INPUT: file data.xlsx (cột câu hỏi dài tiếng Việt)
# Output: in kết quả trực tiếp
# ============================================================

import pandas as pd, numpy as np, re
from scipy import stats
from sklearn.decomposition import PCA

FILE_XLSX = "data_2.xlsx"
N_FACTORS = 4          # chạy theo mô hình lý thuyết RE-EMP-DOC-SAT (đổi 3 nếu theo Kaiser)
MIN_LOADING = 0.50     # ngưỡng loading để đánh dấu biến yếu

# -------------------------
# 0) READ + AUTO RENAME (B1/B2/B3/PHẦN C -> RE/EMP/DOC/SAT)
# -------------------------
df = pd.read_excel(FILE_XLSX, sheet_name=0)
df.columns = [str(c).strip() for c in df.columns]

def pick_cols(pattern: str):
    pat = re.compile(pattern, flags=re.IGNORECASE)
    return [c for c in df.columns if pat.search(str(c))]

b1_cols  = pick_cols(r"\bB1\b")
b2_cols  = pick_cols(r"\bB2\b")
b3_cols  = pick_cols(r"\bB3\b")
sat_cols = pick_cols(r"PHẦN\s*C")

rename_map = {}
for i, c in enumerate(b1_cols,  start=1): rename_map[c] = f"RE{i}"
for i, c in enumerate(b2_cols,  start=1): rename_map[c] = f"EMP{i}"
for i, c in enumerate(b3_cols,  start=1): rename_map[c] = f"DOC{i}"
for i, c in enumerate(sat_cols, start=1): rename_map[c] = f"SAT{i}"

df2 = df.rename(columns=rename_map)
item_cols = list(rename_map.values())

df_items = df2[item_cols].apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)

X = df_items.values
n, p = X.shape
print("EFA data shape (n, p) =", (n, p))
print("Items:", df_items.columns.tolist())

# Standardize => PCA trên ma trận tương quan (giống SPSS)
Xz = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)

# -------------------------
# 1) KMO & Bartlett
# -------------------------
def corr_matrix(xz: np.ndarray) -> np.ndarray:
    return np.corrcoef(xz, rowvar=False)

def kmo_test(R: np.ndarray):
    invR = np.linalg.inv(R)
    D = np.diag(1 / np.sqrt(np.diag(invR)))
    partial = -D @ invR @ D
    np.fill_diagonal(partial, 0.0)

    R2 = R.copy()
    np.fill_diagonal(R2, 0.0)

    r2 = (R2**2).sum()
    p2 = (partial**2).sum()
    kmo_overall = r2 / (r2 + p2)

    r2_j = (R2**2).sum(axis=0)
    p2_j = (partial**2).sum(axis=0)
    kmo_j = r2_j / (r2_j + p2_j)
    return kmo_overall, kmo_j

def bartlett_sphericity(R: np.ndarray, n_samples: int):
    detR = np.linalg.det(R)
    detR = max(detR, 1e-30)  # chống lỗi số
    chi2 = -(n_samples - 1 - (2*p + 5)/6) * np.log(detR)
    df_ = p*(p-1)/2
    pval = 1 - stats.chi2.cdf(chi2, df_)
    return chi2, df_, pval

R = corr_matrix(Xz)
kmo_overall, kmo_per_item = kmo_test(R)
chi2, df_b, p_b = bartlett_sphericity(R, n)

print("\n================= KMO & BARTLETT =================")
print(f"KMO (overall) = {kmo_overall:.3f}")
print(f"Bartlett: chi2={chi2:.3f}, df={df_b:.0f}, p-value={p_b:.6g}")

kmo_item_df = pd.DataFrame({"Item": df_items.columns, "KMO_item": kmo_per_item}).sort_values("KMO_item")
print("\nKMO per item (sorted):")
print(kmo_item_df.to_string(index=False))

# -------------------------
# 2) Eigenvalues (Kaiser > 1)
# -------------------------
pca_all = PCA(n_components=p).fit(Xz)
eigenvalues = pca_all.explained_variance_
eigen_table = pd.DataFrame({"Factor": np.arange(1, p+1), "Eigenvalue": eigenvalues})
n_kaiser = int((eigenvalues > 1).sum())

print("\n================= EIGENVALUES (Kaiser) =================")
print(eigen_table.round(4).to_string(index=False))
print(f"\nGợi ý số nhân tố theo Eigenvalue>1: {n_kaiser}")

# -------------------------
# 3) Varimax rotation + Factor loadings
# -------------------------
def varimax(Phi, gamma=1.0, q=50, tol=1e-6):
    p_, k_ = Phi.shape
    Rm = np.eye(k_)
    d = 0
    for _ in range(q):
        d_old = d
        Lambda = Phi @ Rm
        u, s, vh = np.linalg.svd(
            Phi.T @ (Lambda**3 - (gamma/p_) * Lambda @ np.diag(np.diag(Lambda.T @ Lambda)))
        )
        Rm = u @ vh
        d = s.sum()
        if d_old != 0 and d/d_old < 1 + tol:
            break
    return Phi @ Rm

# PCA loadings (unrotated) cho dữ liệu chuẩn hóa: components_.T * sqrt(eigenvalues)
pca = PCA(n_components=N_FACTORS).fit(Xz)
unrot_loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
rot_loadings = varimax(unrot_loadings)

loadings_df = pd.DataFrame(rot_loadings, index=df_items.columns, columns=[f"F{i+1}" for i in range(N_FACTORS)])

print("\n================= FACTOR LOADINGS (Varimax) =================")
print(loadings_df.round(3).to_string())

# Communalities
communalities = (rot_loadings**2).sum(axis=1)
comm_df = pd.DataFrame({"Item": df_items.columns, "Communality": communalities}).sort_values("Communality")

print("\n================= COMMUNALITIES =================")
print(comm_df.round(3).to_string(index=False))

# -------------------------
# 4) (Tuỳ chọn) đánh dấu biến yếu / tải chéo
# -------------------------
absL = np.abs(rot_loadings)
top1 = absL.max(axis=1)
top2 = np.partition(absL, -2, axis=1)[:, -2] if N_FACTORS >= 2 else np.zeros_like(top1)
gap = top1 - top2

quality = pd.DataFrame({
    "Item": df_items.columns,
    "Top1_loading": top1,
    "Top2_loading": top2,
    "Gap": gap,
    f"Weak_loading(<{MIN_LOADING})": top1 < MIN_LOADING,
    "Cross_loading(Gap<0.20)": gap < 0.20
}).sort_values([f"Weak_loading(<{MIN_LOADING})", "Cross_loading(Gap<0.20)", "Top1_loading"],
               ascending=[False, False, True])

print("\n================= QUALITY CHECK =================")
print(quality.round(3).to_string(index=False))


EFA data shape (n, p) = (75, 20)
Items: ['RE1', 'RE2', 'RE3', 'RE4', 'RE5', 'EMP1', 'EMP2', 'EMP3', 'EMP4', 'EMP5', 'DOC1', 'DOC2', 'DOC3', 'DOC4', 'DOC5', 'SAT1', 'SAT2', 'SAT3', 'SAT4', 'SAT5']

================= KMO & BARTLETT =================
KMO (overall) = 0.911
Bartlett: chi2=1709.126, df=190, p-value=0

KMO per item (sorted):
Item  KMO_item
 RE3  0.814792
SAT2  0.861332
EMP4  0.866195
 RE4  0.880849
DOC2  0.886543
DOC1  0.889099
DOC3  0.891319
EMP3  0.892076
EMP5  0.896901
 RE5  0.902479
DOC5  0.906031
 RE1  0.921392
EMP1  0.935907
SAT1  0.937808
SAT5  0.940569
SAT3  0.941546
 RE2  0.944685
EMP2  0.946729
SAT4  0.948803
DOC4  0.966824

================= EIGENVALUES (Kaiser) =================
 Factor  Eigenvalue
      1     13.4915
      2      1.2678
      3      0.8989
      4      0.7119
      5      0.6427
      6      0.4064
      7      0.3722
      8      0.3531
      9      0.2998
     10      0.2711
     11      0.2471
     12      0.2248
     13      0.1892
     14   

In [39]:
# ============================================================
# 5) EXPORT EFA RESULTS TO EXCEL (GIỐNG SPSS / LUẬN VĂN)
# ============================================================

OUTPUT_XLSX = "EFA_Summary.xlsx"

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:

    # --- Sheet 1: KMO & Bartlett ---
    pd.DataFrame({
        "Statistic": ["KMO (Overall)", "Bartlett Chi-square", "Bartlett df", "Bartlett p-value"],
        "Value": [kmo_overall, chi2, df_b, p_b]
    }).to_excel(writer, sheet_name="KMO_Bartlett", index=False)

    # --- Sheet 2: KMO per item ---
    kmo_item_df.to_excel(writer, sheet_name="KMO_per_item", index=False)

    # --- Sheet 3: Eigenvalues ---
    eigen_table.assign(
        Kaiser=eigen_table["Eigenvalue"] > 1
    ).to_excel(writer, sheet_name="Eigenvalues", index=False)

    # --- Sheet 4: Factor Loadings (Varimax) ---
    loadings_df.round(3).to_excel(writer, sheet_name="Factor_Loadings")

    # --- Sheet 5: Communalities ---
    comm_df.round(3).to_excel(writer, sheet_name="Communalities", index=False)

    # --- Sheet 6: Quality Check ---
    quality.round(3).to_excel(writer, sheet_name="Quality_Check", index=False)

print(f"\n✅ Đã xuất file EFA tổng hợp: {OUTPUT_XLSX}")



✅ Đã xuất file EFA tổng hợp: EFA_Summary.xlsx


In [40]:
# ============================================================
# REGRESSION (OLS) – SPSS-LIKE + EXPORT EXCEL
# ============================================================
# - Auto rename: B1/B2/B3/PHẦN C -> RE/EMP/DOC/SAT
# - Composite mean score
# - OLS multiple regression: SAT ~ RE + EMP + DOC
# - Model fit, Coefficients
# - VIF
# - Simple regression (đối chiếu)
# - Assumption checks (JB, Breusch-Pagan)
# - EXPORT EXCEL (giống SPSS / Trương Viễn Tiên)
# ============================================================

import pandas as pd, numpy as np, re
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

FILE_XLSX   = "data_2.xlsx"
OUTPUT_XLSX = "Regression_Impact_Summary.xlsx"

# ------------------------------------------------------------
# 0) READ + AUTO RENAME
# ------------------------------------------------------------
df = pd.read_excel(FILE_XLSX, sheet_name=0)
df.columns = [str(c).strip() for c in df.columns]

def pick_cols(pattern: str):
    pat = re.compile(pattern, flags=re.IGNORECASE)
    return [c for c in df.columns if pat.search(str(c))]

b1_cols  = pick_cols(r"\bB1\b")
b2_cols  = pick_cols(r"\bB2\b")
b3_cols  = pick_cols(r"\bB3\b")
sat_cols = pick_cols(r"PHẦN\s*C")

rename_map = {}
for i, c in enumerate(b1_cols,  start=1): rename_map[c] = f"RE{i}"
for i, c in enumerate(b2_cols,  start=1): rename_map[c] = f"EMP{i}"
for i, c in enumerate(b3_cols,  start=1): rename_map[c] = f"DOC{i}"
for i, c in enumerate(sat_cols, start=1): rename_map[c] = f"SAT{i}"

df2 = df.rename(columns=rename_map)
item_cols = list(rename_map.values())

df_items = (
    df2[item_cols]
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
    .reset_index(drop=True)
)

print("Valid n =", len(df_items))
print("Items:", df_items.columns.tolist())

# ------------------------------------------------------------
# 1) COMPOSITE MEAN SCORE
# ------------------------------------------------------------
def scale_mean(prefix):
    cols = [c for c in df_items.columns if c.startswith(prefix)]
    return df_items[cols].mean(axis=1)

df_model = pd.DataFrame({
    "RE":  scale_mean("RE"),
    "EMP": scale_mean("EMP"),
    "DOC": scale_mean("DOC"),
    "SAT": scale_mean("SAT"),
})

print("\nComposite head:")
print(df_model.head())

# ------------------------------------------------------------
# 2) OLS MULTIPLE REGRESSION
# ------------------------------------------------------------
y = df_model["SAT"]
X = df_model[["RE", "EMP", "DOC"]]
X_const = sm.add_constant(X)

model = sm.OLS(y, X_const).fit()

coef_table = pd.DataFrame({
    "B": model.params,
    "Std_Error": model.bse,
    "t": model.tvalues,
    "Sig": model.pvalues,
    "CI_Lower": model.conf_int()[0],
    "CI_Upper": model.conf_int()[1],
})

fit_table = pd.DataFrame({
    "N": [len(df_model)],
    "R_squared": [model.rsquared],
    "Adj_R_squared": [model.rsquared_adj],
    "F": [model.fvalue],
    "Sig_F": [model.f_pvalue],
    "AIC": [model.aic],
    "BIC": [model.bic],
    "Durbin_Watson": [sm.stats.stattools.durbin_watson(model.resid)],
})

print("\n================= MODEL FIT =================")
print(fit_table.round(6).to_string(index=False))

print("\n================= COEFFICIENTS =================")
print(coef_table.round(6).to_string())

# ------------------------------------------------------------
# 3) VIF
# ------------------------------------------------------------
vif_df = pd.DataFrame({
    "Variable": X_const.columns,
    "VIF": [variance_inflation_factor(X_const.values, i)
            for i in range(X_const.shape[1])]
})

print("\n================= VIF =================")
print(vif_df.round(3).to_string(index=False))

# ------------------------------------------------------------
# 4) SIMPLE REGRESSION (COMPARISON)
# ------------------------------------------------------------
simple_rows = []
for var in ["RE", "EMP", "DOC"]:
    X1 = sm.add_constant(df_model[[var]])
    m1 = sm.OLS(y, X1).fit()
    simple_rows.append({
        "Model": f"SAT ~ {var}",
        "B": m1.params[var],
        "Sig": m1.pvalues[var],
        "R_squared": m1.rsquared,
        "Adj_R_squared": m1.rsquared_adj
    })

simple_df = pd.DataFrame(simple_rows)

print("\n================= SIMPLE REGRESSION =================")
print(simple_df.round(6).to_string(index=False))

# ------------------------------------------------------------
# 5) ASSUMPTION CHECKS
# ------------------------------------------------------------
jb_stat, jb_p, skew, kurt = sm.stats.stattools.jarque_bera(model.resid)
bp_stat, bp_p, _, _ = het_breuschpagan(model.resid, X_const)

assump_df = pd.DataFrame({
    "Jarque_Bera": [jb_stat],
    "Sig_JB": [jb_p],
    "Skewness": [skew],
    "Kurtosis": [kurt],
    "Breusch_Pagan": [bp_stat],
    "Sig_BP": [bp_p],
})

print("\n================= ASSUMPTION CHECKS =================")
print(assump_df.round(6).to_string(index=False))

# ------------------------------------------------------------
# 6) EXPORT TO EXCEL (SPSS-LIKE)
# ------------------------------------------------------------
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    df_model.round(3).to_excel(writer, sheet_name="Composite_Scores", index=False)
    fit_table.round(6).to_excel(writer, sheet_name="Model_Fit", index=False)
    coef_table.round(6).to_excel(writer, sheet_name="Coefficients")
    vif_df.round(3).to_excel(writer, sheet_name="VIF", index=False)
    simple_df.round(6).to_excel(writer, sheet_name="Simple_Regression", index=False)
    assump_df.round(6).to_excel(writer, sheet_name="Assumption_Checks", index=False)

print(f"\n✅ ĐÃ XUẤT FILE: {OUTPUT_XLSX}")


Valid n = 75
Items: ['RE1', 'RE2', 'RE3', 'RE4', 'RE5', 'EMP1', 'EMP2', 'EMP3', 'EMP4', 'EMP5', 'DOC1', 'DOC2', 'DOC3', 'DOC4', 'DOC5', 'SAT1', 'SAT2', 'SAT3', 'SAT4', 'SAT5']

Composite head:
    RE  EMP  DOC  SAT
0  4.0  3.0  4.6  3.2
1  1.0  1.0  1.0  1.0
2  3.0  2.6  4.0  2.8
3  2.6  2.2  1.2  1.6
4  4.4  3.6  3.6  3.4

================= MODEL FIT =================
 N  R_squared  Adj_R_squared         F  Sig_F        AIC        BIC  Durbin_Watson
75   0.796362       0.787757 92.552556    0.0 121.771715 131.041667       1.969808

================= COEFFICIENTS =================
              B  Std_Error         t       Sig  CI_Lower  CI_Upper
const -1.213400   0.305054 -3.977652  0.000166 -1.821661 -0.605139
RE     0.256124   0.151541  1.690133  0.095389 -0.046040  0.558288
EMP    0.547117   0.149022  3.671392  0.000464  0.249976  0.844259
DOC    0.510810   0.121474  4.205084  0.000075  0.268597  0.753023

================= VIF =================
Variable    VIF
   const 24.756
    

In [23]:
# ============================================================
# MEDIATION ANALYSIS (ĐÃ SỬA LỖI dtype object) – Python
# - RE -> EMP -> SAT
# - RE -> DOC -> SAT
# - Parallel: RE -> (EMP, DOC) -> SAT
# + Bootstrap CI cho indirect effects
# ============================================================

import pandas as pd, numpy as np, re
import statsmodels.api as sm

FILE_XLSX = "data_2.xlsx"
BOOT = 5000
SEED = 42
np.random.seed(SEED)

# -------------------------
# 0) READ + AUTO RENAME -> df_items
# -------------------------
df = pd.read_excel(FILE_XLSX, sheet_name=0)
df.columns = [str(c).strip() for c in df.columns]

def pick_cols(pattern: str):
    pat = re.compile(pattern, flags=re.IGNORECASE)
    return [c for c in df.columns if pat.search(str(c))]

b1_cols  = pick_cols(r"\bB1\b")
b2_cols  = pick_cols(r"\bB2\b")
b3_cols  = pick_cols(r"\bB3\b")
sat_cols = pick_cols(r"PHẦN\s*C")

rename_map = {}
for i, c in enumerate(b1_cols,  start=1): rename_map[c] = f"RE{i}"
for i, c in enumerate(b2_cols,  start=1): rename_map[c] = f"EMP{i}"
for i, c in enumerate(b3_cols,  start=1): rename_map[c] = f"DOC{i}"
for i, c in enumerate(sat_cols, start=1): rename_map[c] = f"SAT{i}"

df2 = df.rename(columns=rename_map)
item_cols = list(rename_map.values())

df_items = df2[item_cols].apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)

# -------------------------
# 1) COMPOSITE MEAN -> df_model
# -------------------------
def scale_mean(prefix):
    cols = [c for c in df_items.columns if str(c).startswith(prefix)]
    return df_items[cols].mean(axis=1)

df_model = pd.DataFrame({
    "RE":  scale_mean("RE"),
    "EMP": scale_mean("EMP"),
    "DOC": scale_mean("DOC"),
    "SAT": scale_mean("SAT"),
}).dropna().reset_index(drop=True)

print("n =", len(df_model))

# -------------------------
# 2) Helpers
# -------------------------
def ols_fit(df_in, y, X_list):
    X = sm.add_constant(df_in[X_list])
    return sm.OLS(df_in[y], X).fit()

def pretty_print_dict(d, title):
    """In dict không bị lỗi dtype object; tự round phần số."""
    rows = []
    for k, v in d.items():
        if isinstance(v, (int, float, np.floating)) and not (pd.isna(v)):
            rows.append((k, float(v)))
        else:
            rows.append((k, v))
    out = pd.DataFrame(rows, columns=["metric", "value"])
    # round chỉ với numeric
    out["value_num"] = pd.to_numeric(out["value"], errors="coerce")
    out.loc[out["value_num"].notna(), "value"] = out.loc[out["value_num"].notna(), "value_num"].round(6)
    out = out.drop(columns=["value_num"])
    print("\n" + "="*18 + f" {title} " + "="*18)
    print(out.to_string(index=False))

def mediation_single_boot(df_in, x, m, y, boot=5000):
    # a: m ~ x
    mod_a = ols_fit(df_in, m, [x])
    a = mod_a.params[x]

    # c: y ~ x
    mod_c = ols_fit(df_in, y, [x])
    c = mod_c.params[x]

    # b & c': y ~ x + m
    mod_b = ols_fit(df_in, y, [x, m])
    b = mod_b.params[m]
    c_prime = mod_b.params[x]

    indirect = a * b

    # bootstrap CI for indirect
    n = len(df_in)
    inds = np.empty(boot, dtype=float)
    for i in range(boot):
        samp = df_in.sample(n=n, replace=True)
        a_b = ols_fit(samp, m, [x]).params[x]
        b_b = ols_fit(samp, y, [x, m]).params[m]
        inds[i] = a_b * b_b

    ci_low, ci_high = np.percentile(inds, [2.5, 97.5])
    sig = (ci_low > 0) or (ci_high < 0)

    res = {
        "a (m~x)": a,
        "b (y~x+m, coef m)": b,
        "c total (y~x)": c,
        "c' direct (y~x+m, coef x)": c_prime,
        "indirect a*b": indirect,
        "boot_CI_low_2.5%": ci_low,
        "boot_CI_high_97.5%": ci_high,
        "indirect_sig(CI excludes 0)": sig,
        "p_a(x)": mod_a.pvalues[x],
        "p_b(m)": mod_b.pvalues[m],
        "p_c_total(x)": mod_c.pvalues[x],
        "p_c_prime(x)": mod_b.pvalues[x],
    }
    return res, mod_a, mod_b, mod_c

def mediation_parallel_boot(df_in, x, mediators, y, boot=5000):
    # a paths
    mods_a = {mm: ols_fit(df_in, mm, [x]) for mm in mediators}
    a = {mm: mods_a[mm].params[x] for mm in mediators}

    # y model: y ~ x + mediators
    mod_y = ols_fit(df_in, y, [x] + mediators)
    b = {mm: mod_y.params[mm] for mm in mediators}
    c_prime = mod_y.params[x]

    # total effect
    mod_c = ols_fit(df_in, y, [x])
    c_total = mod_c.params[x]

    indirect = {mm: a[mm] * b[mm] for mm in mediators}
    indirect_total = sum(indirect.values())

    # bootstrap
    n = len(df_in)
    boot_ind = {mm: np.empty(boot, dtype=float) for mm in mediators}
    boot_total = np.empty(boot, dtype=float)

    for i in range(boot):
        samp = df_in.sample(n=n, replace=True)
        a_b = {mm: ols_fit(samp, mm, [x]).params[x] for mm in mediators}
        mod_y_b = ols_fit(samp, y, [x] + mediators)
        b_b = {mm: mod_y_b.params[mm] for mm in mediators}
        ind_b = {mm: a_b[mm] * b_b[mm] for mm in mediators}
        for mm in mediators:
            boot_ind[mm][i] = ind_b[mm]
        boot_total[i] = sum(ind_b.values())

    def ci(arr):
        lo, hi = np.percentile(arr, [2.5, 97.5])
        sig = (lo > 0) or (hi < 0)
        return lo, hi, sig

    rows = []
    for mm in mediators:
        lo, hi, sig = ci(boot_ind[mm])
        rows.append({
            "effect": f"indirect via {mm}",
            "a": a[mm],
            "b": b[mm],
            "a*b": indirect[mm],
            "CI_low": lo,
            "CI_high": hi,
            "sig(CI excludes 0)": sig
        })

    loT, hiT, sigT = ci(boot_total)
    rows += [
        {"effect": "total indirect (sum)", "a": np.nan, "b": np.nan, "a*b": indirect_total, "CI_low": loT, "CI_high": hiT, "sig(CI excludes 0)": sigT},
        {"effect": "direct c' (x in y~x+M)", "a": np.nan, "b": np.nan, "a*b": c_prime, "CI_low": np.nan, "CI_high": np.nan, "sig(CI excludes 0)": np.nan},
        {"effect": "total c (x in y~x)", "a": np.nan, "b": np.nan, "a*b": c_total, "CI_low": np.nan, "CI_high": np.nan, "sig(CI excludes 0)": np.nan},
    ]

    return pd.DataFrame(rows), mods_a, mod_y, mod_c

# -------------------------
# 3) MEDIATION ĐƠN
# -------------------------
res_emp, mod_a_emp, mod_b_emp, mod_c_emp = mediation_single_boot(df_model, "RE", "EMP", "SAT", boot=BOOT)
pretty_print_dict(res_emp, "MEDIATION: RE -> EMP -> SAT")

res_doc, mod_a_doc, mod_b_doc, mod_c_doc = mediation_single_boot(df_model, "RE", "DOC", "SAT", boot=BOOT)
pretty_print_dict(res_doc, "MEDIATION: RE -> DOC -> SAT")

# -------------------------
# 4) PARALLEL MEDIATION
# -------------------------
table_parallel, mods_a, mod_y, mod_c = mediation_parallel_boot(df_model, "RE", ["EMP", "DOC"], "SAT", boot=BOOT)

print("\n" + "="*18 + " PARALLEL MEDIATION: RE -> (EMP, DOC) -> SAT " + "="*18)
# round chỉ các cột số
num_cols = ["a","b","a*b","CI_low","CI_high"]
table_parallel[num_cols] = table_parallel[num_cols].apply(pd.to_numeric, errors="coerce").round(6)
print(table_parallel.to_string(index=False))

# (Tuỳ chọn) nếu muốn xem summary mô hình:
# print(mod_a_emp.summary())  # EMP ~ RE
# print(mod_b_emp.summary())  # SAT ~ RE + EMP
# print(mod_y.summary())      # SAT ~ RE + EMP + DOC


n = 75

================== MEDIATION: RE -> EMP -> SAT ==================
                     metric     value
                    a (m~x)  0.888375
          b (y~x+m, coef m)  0.827994
              c total (y~x)   1.19999
  c' direct (y~x+m, coef x)  0.464421
               indirect a*b  0.735569
           boot_CI_low_2.5%  0.462335
         boot_CI_high_97.5%  1.062815
indirect_sig(CI excludes 0)       1.0
                     p_a(x)       0.0
                     p_b(m)       0.0
               p_c_total(x)       0.0
               p_c_prime(x)  0.004645

================== MEDIATION: RE -> DOC -> SAT ==================
                     metric     value
                    a (m~x)  0.896264
          b (y~x+m, coef m)  0.710707
              c total (y~x)   1.19999
  c' direct (y~x+m, coef x)  0.563009
               indirect a*b  0.636981
           boot_CI_low_2.5%   0.42894
         boot_CI_high_97.5%  0.879698
indirect_sig(CI excludes 0)       1.0
                     p_

In [42]:
# ============================================================
# MEDIATION ANALYSIS (ĐÃ SỬA LỖI dtype object) – Python
# - RE -> EMP -> SAT
# - RE -> DOC -> SAT
# - Parallel: RE -> (EMP, DOC) -> SAT
# + Bootstrap CI cho indirect effects
# ============================================================

import pandas as pd, numpy as np, re
import statsmodels.api as sm

FILE_XLSX = "data_2.xlsx"
BOOT = 5000
SEED = 42
np.random.seed(SEED)

# -------------------------
# 0) READ + AUTO RENAME -> df_items
# -------------------------
df = pd.read_excel(FILE_XLSX, sheet_name=0)
df.columns = [str(c).strip() for c in df.columns]

def pick_cols(pattern: str):
    pat = re.compile(pattern, flags=re.IGNORECASE)
    return [c for c in df.columns if pat.search(str(c))]

b1_cols  = pick_cols(r"\bB1\b")
b2_cols  = pick_cols(r"\bB2\b")
b3_cols  = pick_cols(r"\bB3\b")
sat_cols = pick_cols(r"PHẦN\s*C")

rename_map = {}
for i, c in enumerate(b1_cols,  start=1): rename_map[c] = f"RE{i}"
for i, c in enumerate(b2_cols,  start=1): rename_map[c] = f"EMP{i}"
for i, c in enumerate(b3_cols,  start=1): rename_map[c] = f"DOC{i}"
for i, c in enumerate(sat_cols, start=1): rename_map[c] = f"SAT{i}"

df2 = df.rename(columns=rename_map)
item_cols = list(rename_map.values())

df_items = df2[item_cols].apply(pd.to_numeric, errors="coerce").dropna().reset_index(drop=True)

# -------------------------
# 1) COMPOSITE MEAN -> df_model
# -------------------------
def scale_mean(prefix):
    cols = [c for c in df_items.columns if str(c).startswith(prefix)]
    return df_items[cols].mean(axis=1)

df_model = pd.DataFrame({
    "RE":  scale_mean("RE"),
    "EMP": scale_mean("EMP"),
    "DOC": scale_mean("DOC"),
    "SAT": scale_mean("SAT"),
}).dropna().reset_index(drop=True)

print("n =", len(df_model))

# -------------------------
# 2) Helpers
# -------------------------
def ols_fit(df_in, y, X_list):
    X = sm.add_constant(df_in[X_list])
    return sm.OLS(df_in[y], X).fit()

def pretty_print_dict(d, title):
    """In dict không bị lỗi dtype object; tự round phần số."""
    rows = []
    for k, v in d.items():
        if isinstance(v, (int, float, np.floating)) and not (pd.isna(v)):
            rows.append((k, float(v)))
        else:
            rows.append((k, v))
    out = pd.DataFrame(rows, columns=["metric", "value"])
    # round chỉ với numeric
    out["value_num"] = pd.to_numeric(out["value"], errors="coerce")
    out.loc[out["value_num"].notna(), "value"] = out.loc[out["value_num"].notna(), "value_num"].round(6)
    out = out.drop(columns=["value_num"])
    print("\n" + "="*18 + f" {title} " + "="*18)
    print(out.to_string(index=False))

def mediation_single_boot(df_in, x, m, y, boot=5000):
    # a: m ~ x
    mod_a = ols_fit(df_in, m, [x])
    a = mod_a.params[x]

    # c: y ~ x
    mod_c = ols_fit(df_in, y, [x])
    c = mod_c.params[x]

    # b & c': y ~ x + m
    mod_b = ols_fit(df_in, y, [x, m])
    b = mod_b.params[m]
    c_prime = mod_b.params[x]

    indirect = a * b

    # bootstrap CI for indirect
    n = len(df_in)
    inds = np.empty(boot, dtype=float)
    for i in range(boot):
        samp = df_in.sample(n=n, replace=True)
        a_b = ols_fit(samp, m, [x]).params[x]
        b_b = ols_fit(samp, y, [x, m]).params[m]
        inds[i] = a_b * b_b

    ci_low, ci_high = np.percentile(inds, [2.5, 97.5])
    sig = (ci_low > 0) or (ci_high < 0)

    res = {
        "a (m~x)": a,
        "b (y~x+m, coef m)": b,
        "c total (y~x)": c,
        "c' direct (y~x+m, coef x)": c_prime,
        "indirect a*b": indirect,
        "boot_CI_low_2.5%": ci_low,
        "boot_CI_high_97.5%": ci_high,
        "indirect_sig(CI excludes 0)": sig,
        "p_a(x)": mod_a.pvalues[x],
        "p_b(m)": mod_b.pvalues[m],
        "p_c_total(x)": mod_c.pvalues[x],
        "p_c_prime(x)": mod_b.pvalues[x],
    }
    return res, mod_a, mod_b, mod_c

def mediation_parallel_boot(df_in, x, mediators, y, boot=5000):
    # a paths
    mods_a = {mm: ols_fit(df_in, mm, [x]) for mm in mediators}
    a = {mm: mods_a[mm].params[x] for mm in mediators}

    # y model: y ~ x + mediators
    mod_y = ols_fit(df_in, y, [x] + mediators)
    b = {mm: mod_y.params[mm] for mm in mediators}
    c_prime = mod_y.params[x]

    # total effect
    mod_c = ols_fit(df_in, y, [x])
    c_total = mod_c.params[x]

    indirect = {mm: a[mm] * b[mm] for mm in mediators}
    indirect_total = sum(indirect.values())

    # bootstrap
    n = len(df_in)
    boot_ind = {mm: np.empty(boot, dtype=float) for mm in mediators}
    boot_total = np.empty(boot, dtype=float)

    for i in range(boot):
        samp = df_in.sample(n=n, replace=True)
        a_b = {mm: ols_fit(samp, mm, [x]).params[x] for mm in mediators}
        mod_y_b = ols_fit(samp, y, [x] + mediators)
        b_b = {mm: mod_y_b.params[mm] for mm in mediators}
        ind_b = {mm: a_b[mm] * b_b[mm] for mm in mediators}
        for mm in mediators:
            boot_ind[mm][i] = ind_b[mm]
        boot_total[i] = sum(ind_b.values())

    def ci(arr):
        lo, hi = np.percentile(arr, [2.5, 97.5])
        sig = (lo > 0) or (hi < 0)
        return lo, hi, sig

    rows = []
    for mm in mediators:
        lo, hi, sig = ci(boot_ind[mm])
        rows.append({
            "effect": f"indirect via {mm}",
            "a": a[mm],
            "b": b[mm],
            "a*b": indirect[mm],
            "CI_low": lo,
            "CI_high": hi,
            "sig(CI excludes 0)": sig
        })

    loT, hiT, sigT = ci(boot_total)
    rows += [
        {"effect": "total indirect (sum)", "a": np.nan, "b": np.nan, "a*b": indirect_total, "CI_low": loT, "CI_high": hiT, "sig(CI excludes 0)": sigT},
        {"effect": "direct c' (x in y~x+M)", "a": np.nan, "b": np.nan, "a*b": c_prime, "CI_low": np.nan, "CI_high": np.nan, "sig(CI excludes 0)": np.nan},
        {"effect": "total c (x in y~x)", "a": np.nan, "b": np.nan, "a*b": c_total, "CI_low": np.nan, "CI_high": np.nan, "sig(CI excludes 0)": np.nan},
    ]

    return pd.DataFrame(rows), mods_a, mod_y, mod_c

# -------------------------
# 3) MEDIATION ĐƠN
# -------------------------
res_emp, mod_a_emp, mod_b_emp, mod_c_emp = mediation_single_boot(df_model, "RE", "EMP", "SAT", boot=BOOT)
pretty_print_dict(res_emp, "MEDIATION: RE -> EMP -> SAT")

res_doc, mod_a_doc, mod_b_doc, mod_c_doc = mediation_single_boot(df_model, "RE", "DOC", "SAT", boot=BOOT)
pretty_print_dict(res_doc, "MEDIATION: RE -> DOC -> SAT")

# -------------------------
# 4) PARALLEL MEDIATION
# -------------------------
table_parallel, mods_a, mod_y, mod_c = mediation_parallel_boot(df_model, "RE", ["EMP", "DOC"], "SAT", boot=BOOT)

print("\n" + "="*18 + " PARALLEL MEDIATION: RE -> (EMP, DOC) -> SAT " + "="*18)
# round chỉ các cột số
num_cols = ["a","b","a*b","CI_low","CI_high"]
table_parallel[num_cols] = table_parallel[num_cols].apply(pd.to_numeric, errors="coerce").round(6)
print(table_parallel.to_string(index=False))

# (Tuỳ chọn) nếu muốn xem summary mô hình:
# print(mod_a_emp.summary())  # EMP ~ RE
# print(mod_b_emp.summary())  # SAT ~ RE + EMP
# print(mod_y.summary())      # SAT ~ RE + EMP + DOC


n = 75

================== MEDIATION: RE -> EMP -> SAT ==================
                     metric     value
                    a (m~x)  0.888375
          b (y~x+m, coef m)  0.827994
              c total (y~x)   1.19999
  c' direct (y~x+m, coef x)  0.464421
               indirect a*b  0.735569
           boot_CI_low_2.5%  0.462335
         boot_CI_high_97.5%  1.062815
indirect_sig(CI excludes 0)       1.0
                     p_a(x)       0.0
                     p_b(m)       0.0
               p_c_total(x)       0.0
               p_c_prime(x)  0.004645

================== MEDIATION: RE -> DOC -> SAT ==================
                     metric     value
                    a (m~x)  0.896264
          b (y~x+m, coef m)  0.710707
              c total (y~x)   1.19999
  c' direct (y~x+m, coef x)  0.563009
               indirect a*b  0.636981
           boot_CI_low_2.5%   0.42894
         boot_CI_high_97.5%  0.879698
indirect_sig(CI excludes 0)       1.0
                     p_

In [43]:
# ============================================================
# 5) EXPORT MEDIATION RESULTS TO EXCEL (SPSS-like)
# ============================================================

OUTPUT_FILE = "Mediation_Results.xlsx"

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:

    # ---------
    # Sheet 1: Mediation RE -> EMP -> SAT
    # ---------
    df_emp = pd.DataFrame.from_dict(res_emp, orient="index", columns=["Value"])
    df_emp.index.name = "Metric"
    df_emp.reset_index(inplace=True)
    df_emp.to_excel(writer, sheet_name="RE_EMP_SAT", index=False)

    # ---------
    # Sheet 2: Mediation RE -> DOC -> SAT
    # ---------
    df_doc = pd.DataFrame.from_dict(res_doc, orient="index", columns=["Value"])
    df_doc.index.name = "Metric"
    df_doc.reset_index(inplace=True)
    df_doc.to_excel(writer, sheet_name="RE_DOC_SAT", index=False)

    # ---------
    # Sheet 3: Parallel mediation RE -> (EMP, DOC) -> SAT
    # ---------
    table_parallel.to_excel(writer, sheet_name="Parallel_Mediation", index=False)

print(f"\n✅ Đã xuất file Excel: {OUTPUT_FILE}")



✅ Đã xuất file Excel: Mediation_Results.xlsx
